# AIC25 Single & Multi-Camera Tracking Pipeline
Works both **locally in VS Code** and on **Google Colab**.
Run cells top to bottom — the first cell sets everything up automatically.

---
## 0. Environment setup — runs automatically for local or Colab

In [1]:
import os, sys

# ── Detect environment ────────────────────────────────────────────────────────
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    REPO = '/content/repo'
    PYTHON = 'python'
    print('Running on Google Colab')
else:
    REPO = '/home/seco/deepLearning/Single-Camera-Tracking-Consistency'
    PYTHON = f'{REPO}/.venv/bin/python'
    print('Running locally in VS Code')

os.chdir(REPO)
print(f'REPO  : {REPO}')
print(f'PYTHON: {PYTHON}')

# ── GPU check ────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run([PYTHON, '-c', 'import torch; print("CUDA:", torch.cuda.is_available()); print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — CPU only")'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr[:300])

Running locally in VS Code
REPO  : /home/seco/deepLearning/Single-Camera-Tracking-Consistency
PYTHON: /home/seco/deepLearning/Single-Camera-Tracking-Consistency/.venv/bin/python
CUDA: False
GPU: None — CPU only



In [2]:
# ── Colab only: clone repo + install deps ─────────────────────────────────────
if ON_COLAB:
    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
    else:
        os.system(f'git -C {REPO} pull')

    os.chdir(f'{REPO}/BoT-SORT')
    os.system('pip install -q -r requirements.txt')
    os.system('pip install -q cython_bbox pycocotools faiss-gpu cmake')
    os.system('python setup.py develop --quiet')

    os.chdir(f'{REPO}/deep-person-reid')
    os.system('pip install -q -r requirements.txt')
    os.system('python setup.py develop --quiet')

    os.chdir(REPO)
    os.system('pip install -q -r tracking/requirements.txt')
    print('Colab deps installed.')
else:
    print('Local: skipping install (using .venv)')

Local: skipping install (using .venv)


In [3]:
# ── Colab only: download models ───────────────────────────────────────────────
if ON_COLAB:
    os.system('pip install -q gdown')

    osnet = f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar'
    os.makedirs(os.path.dirname(osnet), exist_ok=True)
    if not os.path.exists(osnet):
        os.system(f'gdown "https://drive.google.com/uc?id=1IosIFlLiulGIjwW3H8uMCC3YvMyr9gZ2" -O {osnet}')

    bytetrack = f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'
    os.makedirs(os.path.dirname(bytetrack), exist_ok=True)
    if not os.path.exists(bytetrack):
        os.system(f'gdown "https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5" -O {bytetrack}')

    # Copy ai_city_ckpt from Drive if available
    from google.colab import drive
    drive.mount('/content/drive')
    ai_src = '/content/drive/MyDrive/AIC25/models/ai_city_ckpt.pth.tar'
    ai_dst = f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'
    if os.path.exists(ai_src):
        os.system(f'cp {ai_src} {ai_dst}')
        print('ai_city_ckpt.pth.tar copied from Drive.')
    else:
        print('ai_city_ckpt not found on Drive — use bytetrack_x_mot17.pth.tar for now.')
else:
    print('Local: models already in place.')

Local: models already in place.


---
## 1. Configure scene
Change `SCENE` here to run a different warehouse.

In [4]:
SCENE   = 'Warehouse_016'
DATASET = 'Val'           # 'Val' or 'Train'

# Verify data exists
data_dir = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
print('Data dir exists:', os.path.exists(data_dir))
if os.path.exists(data_dir):
    print('Contents:', os.listdir(data_dir))

Data dir exists: True
Contents: ['depth_map', 'videos', 'calibration.json', 'ground_truth.json']


---
## 2. Download dataset — Colab only
**Skip this section when running locally** — data is already in `AIC25_Track1/`.

In [5]:
if ON_COLAB:
    os.system('pip install -q huggingface_hub')
    from huggingface_hub import snapshot_download
    import shutil

    hf_split = DATASET.lower()
    snapshot_download(
        repo_id='nvidia/PhysicalAI-SmartSpaces',
        repo_type='dataset',
        local_dir='/content/hf_data',
        allow_patterns=[
            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/videos/**',
            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/calibration.json',
            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/ground_truth.json',
        ]
    )

    HF   = '/content/hf_data/MTMC_Tracking_2025'
    DEST = f'{REPO}/AIC25_Track1'
    src  = f'{HF}/{hf_split}/{SCENE}'
    dst  = f'{DEST}/{DATASET}/{SCENE}'
    os.makedirs(dst, exist_ok=True)

    for fname in ['calibration.json', 'ground_truth.json']:
        if os.path.exists(f'{src}/{fname}'):
            shutil.copy(f'{src}/{fname}', f'{dst}/{fname}')

    if not os.path.exists(f'{dst}/videos'):
        os.symlink(f'{src}/videos', f'{dst}/videos')

    if os.path.exists(f'{src}/depth_maps') and not os.path.exists(f'{dst}/depth_map'):
        os.symlink(f'{src}/depth_maps', f'{dst}/depth_map')

    os.makedirs(f'{dst}/depth_map', exist_ok=True)
    print('Dataset ready.')
else:
    print('Local: using existing AIC25_Track1/ data.')

Local: using existing AIC25_Track1/ data.


---
## [B] Extract frames from video

In [6]:
os.chdir(REPO)
ret = os.system(f'{PYTHON} tools/extract_frames_25.py ./AIC25_Track1/{DATASET} -s {SCENE}')
print('Done' if ret == 0 else f'ERROR code {ret}')

[Processing] Warehouse_016 Camera
./AIC25_Track1/Val/Warehouse_016/videos/Camera/Frame already exists. If you want to regenrate, manually remove it.[Processing] Warehouse_016 Camera_03
./AIC25_Track1/Val/Warehouse_016/videos/Camera_03/Frame already exists. If you want to regenrate, manually remove it.[Processing] Warehouse_016 Camera_08
./AIC25_Track1/Val/Warehouse_016/videos/Camera_08/Frame already exists. If you want to regenrate, manually remove it.[Processing] Warehouse_016 Camera_10
[Processing] Warehouse_016 Camera_05
./AIC25_Track1/Val/Warehouse_016/videos/Camera_05/Frame already exists. If you want to regenrate, manually remove it../AIC25_Track1/Val/Warehouse_016/videos/Camera_10/Frame already exists. If you want to regenrate, manually remove it.[Processing] Warehouse_016 Camera_06
./AIC25_Track1/Val/Warehouse_016/videos/Camera_06/Frame already exists. If you want to regenrate, manually remove it.[Processing] Warehouse_016 Camera_07
./AIC25_Track1/Val/Warehouse_016/videos/Camer

---
## [C] Detection — YOLOX
Tip: add `--camera Camera --max_frames 100` to test on 1 camera / 100 frames first.

In [ ]:
os.chdir(f'{REPO}/BoT-SORT')

# Full run (all cameras, all frames):
cmd = f'{PYTHON} tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} ../'

# Quick test (1 camera, 100 frames) — uncomment to use:
# cmd = f'{PYTHON} tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} --camera Camera --max_frames 100 ../'

print('Running:', cmd)
ret = os.system(cmd)
print('Done' if ret == 0 else f'ERROR code {ret}')

Running: /home/seco/deepLearning/Single-Camera-Tracking-Consistency/.venv/bin/python tools/aic25_get_detection.py --scene Warehouse_016 --dataset Val ../


---
## [D] Re-ID embeddings — OSNet

In [ ]:
os.chdir(f'{REPO}/deep-person-reid')
ret = os.system(f'{PYTHON} torchreid/aic25_extract.py -s {SCENE} ../')
print('Done' if ret == 0 else f'ERROR code {ret}')

---
## [E] Single-camera tracking — all cameras

In [ ]:
os.chdir(REPO)
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted([d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}')])
print(f'{len(cameras)} cameras: {cameras}')

for cam in cameras:
    print(f'\n=== {cam} ===')
    ret = os.system(f'{PYTHON} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('OK' if ret == 0 else f'ERROR {ret}')

---
## [F] Fix single-camera results

In [ ]:
os.chdir(REPO)
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted([d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}')])

for cam in cameras:
    print(f'\n=== Fixing {cam} ===')
    ret = os.system(f'{PYTHON} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('OK' if ret == 0 else f'ERROR {ret}')

---
## [G] Multi-camera tracking

In [ ]:
os.chdir(REPO)
ret = os.system(f'{PYTHON} BoT-SORT/multi_camera_revised.py -s {SCENE} --dataset {DATASET}')
print('Done' if ret == 0 else f'ERROR {ret}')

In [ ]:
os.chdir(REPO)
ret = os.system(f'{PYTHON} BoT-SORT/multi_camera_fix.py -s {SCENE} --dataset {DATASET}')
print('Done' if ret == 0 else f'ERROR {ret}')

---
## [H] Evaluate

In [ ]:
os.chdir(REPO)
ret = os.system(f'{PYTHON} TrackEval/main.py')
print('Done' if ret == 0 else f'ERROR {ret}')

---
## Check outputs

In [ ]:
import json

# Show single-camera tracking result for Camera
fixed = f'{REPO}/Tracking/Singlecamera/{SCENE}/Camera/fixed_Camera.json'
if os.path.exists(fixed):
    with open(fixed) as f:
        data = json.load(f)
    all_ids = set()
    for tracks in data.values():
        for t in tracks:
            all_ids.add(t.get('object sc id'))
    print(f'Frames tracked: {len(data)}')
    print(f'Unique track IDs: {sorted(all_ids)}')
else:
    print('No output yet — run steps B through F first.')